In [1]:
import fastf1 as f1
import pandas as pd
import sqlite3
from pathlib import Path

In [2]:
pd.set_option('display.max_columns', None)

In [ ]:
def print_info(level, message):
    string = "knapsack"

    if level.upper() == 'INFO':
        string += "       INFO     "
    if level.upper() == 'WARNING':
        string += "    WARNING     "
    if level.upper() == 'ERROR':
        string += "      ERROR     "

    string += message
    print(string)

In [4]:
# Database Setup
conn = sqlite3.connect("f1_fantasy.db")
cursor = conn.cursor()

def run_sql_script(connection, path):
    sql = Path(path).read_text()
    connection.executescript(sql)
    conn.commit()

In [5]:
cursor.execute("""
INSERT INTO team_aliases (team_id, alias) VALUES (5, 'Renault')
""")
conn.commit()

In [6]:
# Build initial schema

run_sql_script(conn, "sql/initial_schema.sql")

In [7]:
# Insert team data
try:
    run_sql_script(conn, "sql/teams_insert.sql")
except sqlite3.IntegrityError as e:
    print("Failed to insert: Teams already inserted.")

Failed to insert: Teams already inserted.


In [8]:
# Fetch team data
cursor.execute("SELECT canonical FROM teams;")
TEAMS = [row[0] for row in cursor.fetchall()]
TEAMS

['Ferrari',
 'Mercedes',
 'Red Bull',
 'McLaren',
 'Alpine',
 'Aston Martin',
 'Williams',
 'Audi',
 'Haas',
 'Racing Bulls']

In [9]:
# Insert team aliases
try:
    run_sql_script(conn, "sql/team_aliases_insert.sql")
except sqlite3.IntegrityError as e:
    print("Failed to insert: Team aliases already inserted.")

Failed to insert: Team aliases already inserted.


In [13]:
YEARS = sorted([2014, 2015, 2017, 2018, 2022, 2023], reverse=True)
FAIL_FOR_TEAM_MISSING = False

class MissingTeamError(Exception):
    pass

for year in YEARS:
    schedule = f1.get_event_schedule(year)
    num_rounds_in_year = len(schedule[schedule['RoundNumber'] != 0])

    try:
        cursor.execute("""
        INSERT INTO seasons VALUES (?, ?)
        """, (year, num_rounds_in_year))
        conn.commit()
    except sqlite3.IntegrityError as e:
        print_info('WARNING', 'Failed to insert - Year already inserted, continuing execution...')
    except Exception as e:
        raise e

    for round_number in range(1, num_rounds_in_year + 1):
        cursor.execute("""
        SELECT COUNT(*)
        FROM session_points_history
        WHERE year = ? AND round_number = ?;
        """, (year, round_number))

        row = cursor.fetchone()
        print(row[0], len(TEAMS))

        if row[0] >= len(TEAMS) - 1:
            print_info('INFO', 'Already have all team points info for this session, continuing execution...')
            continue

        event = f1.get_event(year, round_number)
        session = event.get_session(5)
        session.load()
        present_names = set(session.results['TeamName'])

        try:
            cursor.execute("""
            INSERT INTO events VALUES (?, ?)
            """, (year, round_number))
            conn.commit()
        except sqlite3.IntegrityError as e:
            print_info('WARNING', 'Failed to insert - Round already inserted, continuing execution...')
        except Exception as e:
            raise e

        for team in TEAMS:
            try:
                cursor.execute("""
                SELECT alias, t.team_id
                FROM team_aliases ta
                JOIN teams t ON ta.team_id = t.team_id
                WHERE t.canonical = ?;
                """, (team,))
                taids = [(row[0], row[1]) for row in cursor.fetchall()]
                team_aliases = []
                team_id = taids[0][1]
                for tup in taids:
                    team_aliases.append(tup[0])
                    

            except Exception as e:
                print_info('ERROR', 'Failed query for team names - {e}')
                raise e

            try:
                if not any(name in present_names for name in team_aliases):
                    raise MissingTeamError(f'Team names {team_aliases} not found in session. Potential names include: {set(session.results['TeamName'])}\n')

                team_session_points = session.results.loc[
                    session.results['TeamName'].isin(team_aliases), 'Points'
                ].sum()

                cursor.execute("""
                INSERT INTO session_points_history (year, round_number, team_id, points)
                VALUES
                    (?, ?, ?, ?);
                """, (year, round_number, team_id, team_session_points))
                conn.commit()
                print_info('INFO', f"Stored points for {team} in round {round_number} of {year}")

            except sqlite3.IntegrityError as e:
                if 'UNIQUE' in str(e).split():
                    print_info('INFO', f"Already have stored points for {team} in round {round_number} of {year}")
                    continue

            except MissingTeamError as e:
                print_info('ERROR', str(e))

                if FAIL_FOR_TEAM_MISSING:
                    raise MissingTeamError(e)

                continue

            except Exception as e:
                print_info('ERROR', f"Failed to fetch data: {e}")
                conn.rollback()
                raise e

knapsack    WARNING     Failed to insert - Year already inserted, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points info for this session, continuing execution...
10 10
knapsack       INFO     Already have all team points inf